# TP 2 : Analyse d'un LLM et benchmarking

## 1. Analyse d'un LLM en local

### 1.1 Installation

#### 1.1.1 Configuration de VS Code

Voir `tp2.pptx` sur Moodle.

#### 1.1.2 Installation des librairies

In [ ]:
# Installation des bibliothèques  (Ne faire qu'une fois)
%pip install transformers accelerate huggingface_hub
%pip install ipywidgets

: 

#### 1.1.3 Vérification de l'environnement



In [2]:
# Vérification de l'environnement
import torch
import transformers

print(f"PyTorch version   : {torch.__version__}")
print(f"Transformers ver  : {transformers.__version__}")
print(f"CUDA disponible   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM              : {total_mem:.1f} GB")
else:
    print("⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)")

PyTorch version   : 2.10.0
Transformers ver  : 5.3.0
CUDA disponible   : False
⚠️  Pas de GPU détecté — le CPU sera utilisé (plus lent)


### 1.2 Qwen

#### 1.2.1 Qwen/Qwen3-0.6B-Base

Nous étudions d'abord un modèle LLM brut, dont l'identifiant modèle est : `Qwen/Qwen3-0.6B-Base`

1. Expliquer les différents parties de l'identifiant. Pourquoi utilisons-nous la version `0.6B` dans ce TP ?

> Il y a quatres partie principales : **Qwen** (organisation ou namespace), **Qwen3** (génération version de modèle), **0.6B** (taille - nombre de paramètres), **Base** (type/variante du modèle). On utilise 0.6B à cause de la limitation du nombre de paramètres (Aide de ChatGPT)

2. Tester le code ci-dessous. Que fait-il ?

> Le code :
1️⃣ charge la librairie ML, 2️⃣ télécharge le tokenizer du modèle, 3️⃣ télécharge le modèle LLM, 4️⃣ configure l’exécution sur GPU Apple, 5️⃣ crée une fonction simple de génération de texte

3. Commenter chaque ligne du code pour comprendre son rôle.

4. Tester différents prompts.

> Les réponses ne sont pas très cohérentes

5. Changer le paramètre `temperature`. Qu'est-ce que vous observez ? Que fait concrètement ce paramètre ? (température varie entre 0 et 2 ?)

> Plus la température est élevée, plus il est créatif - c'est la propa de quel "mot serrait le plus probable" après un mot - ajout d'aléatoire


6. Et si vous remplacez l'argument `temperature` par `do_sample=False` et que vous exécutez le bloc le deux fois de suite ?

> ça se répète

In [ ]:
from transformers import pipeline, AutoTokenizer # import de la librairie Transformers 2 outils : pipeline et AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [45]:
prompt = "Olà esta tudo bem ?"
# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=1.9, # Combien d'aléatoire peut générer - prédiction de mots suivent - calculs la propabilité du mots le plus probable qui va suivre
    
    max_new_tokens=120,
    return_full_text=True,
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Olà esta tudo bem ? Porque quase que acredere a todos no sistema operacional?

Parece o suficiente que é bom, por ser bem pensado e pensado bem... Apenas um simples resumidário... 

Parece ter a certeza de todas os dados de acesso em todos os computadores, o mundo e as redes? Não. Isso parece estar em excesso.

O que acredere no que, quando for bem pensado e pensado... O mesmo.

Por isso agradecemos. A resposta é "Sociedade" (no sentido


#### 1.2.2 Qwen/Qwen3-0.6B

Nous utilisons maintenant la version sans `Base` de `Qwen3-0.6B-Base`.

1. Que signifie ce changement dans l'identifiant ?

> Réponse de ChatGPT : La variante du modèle. 
>
> Elle a subi un fine-tuning supplémentaire : Instruction tuning, RLHF ou alignement, données de conversation, dialoguer avec l’utilisateur
>
> Donc elle est entraînée à : répondre aux questions et suivre des instructions
=> on passe d’un modèle pré-entraîné brut (Base) à un modèle fine-tuned pour suivre des instructions (Adapté à répondre à des questions)


2. Tester le code ci-dessous. Qu'est-ce que est différent ?

> Pas de répétions dans la questions - réponses plus claires et structurées

3. Commenter les nouvelles lignes pour expliquer leurs rôles ?

>

3. Tester différents `content` pour `user`.
4. Décommenter les lignes `system`. Tester différents `content`. 
5. Analysez le contenu du prompt. Quels sont ses différents éléments et que veulent-ils dire ?

> Les modèles de chat ne reçoivent pas simplement du texte brut.
Ils attendent un format de conversation spécifique (appelé chat template).

>Cette fonction :
>prend une conversation structurée (messages),
>applique le format attendu par le modèle,
>produit le prompt final

>Donc : messages (structure Python)
↓
apply_chat_template
↓
prompt formaté pour le LLM

6. Et que se passe-t-il si vous remplacez le `prompt` par une question ou un texte quelconque ?

> compléter le prompt, fait pour répondre à la question ?

7. Que se passe-t-il avec `enable_thinking=True` ?

> Les réponses sont que en anglais, pense mieux, j'ai l'impression

In [20]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device="mps",
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [64]:
# ---- Début des nouvelles lignes
messages = [
    # {"role": "system", "content": "You only answer with one word."},
    {"role": "system", "content": "You have to answer in French."},
    {"role": "user",   "content": "Peux-tu me fournir une recette pour des crêpes ?"}
    #{"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

# prompt = "Ainsi donc"
# ---- Fin des nouvelles lignes

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=120,
    return_full_text=False, # ne pas retourner pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Sure, I can provide you with a recipe for homemade crêpes:

Ingredients:
- 1 cup all-purpose flour
- 2 tbsp granulated sugar
- 1 tbsp baking powder
- 1/2 tsp salt
- 1 cup milk
- 1 egg
- 1/2 cup vegetable oil
- 1 tbsp vanilla extract
- 1/2 cup powdered sugar
- 1/4 cup milk

Inst


### 1.3 TinyLlama

Voici un autre modèle : `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

1. Tester quelques questions et comparer les réponses à celles de Qwen

***Qwen*** : Sure! Here's a simple recipe for homemade crêpes:
Ingredients:
- 3 cups all-purpose flour
- 1 tsp baking powder
- 1 tsp salt
- 1/2 cup unsalted butter, cold and cubed
- 2 large eggs
- 1 cup whole milk
- 1/2 cup white sugar
- 1 tbsp vanilla extract

Instructions:
1. In a large bowl, whisk

***TinyLlama*** :

Certainement! Nous avons des recettes pour des crêpes sur notre site. Une recette simple en français pour des crêpes est disponible ici: https://www.julierobot.com/en/recipes/simple-french-crêpes/




2. Tester différents paramètres.
2. Que pensez-vous de la capacité de TinyLlama à suivre les instructions `system`?

> Il les suit mieux que qwen mais ça dépend pour quelles raisons.

In [49]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [65]:
messages = [
    #{"role": "system", "content": "You only answer with one word."},
    # {"role": "system", "content": "You answer with a json."},
    {"role": "system", "content": "You have to answer in French."},
    {"role": "user",   "content": "Peux-tu me fournir une recette pour des crêpes ?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=1.0,
    max_new_tokens=1000, # max de mot que peux founir je crois 
    return_full_text=False, # ne retourne pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

Both `max_new_tokens` (=1000) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



RÉPONSE

Certain, je peux vous guider dans la recette de crêpes sans cacété.

Crêpe recipe in French (recipe sans alcool):

Crêpes

Dur au plomb, 460 gr pour 6 pincées.

1. Éclailler les grains d'avoine. Aucun cacété n'est insalubre, mais il convient de les étaler par-dessus le jus de l'œuf et du beurre sans les conserver, avant de les incorporer à la blender pour s'amorcer avec une machine à faire et la poacher dans le four à long-entière. Le cacélimé va passer par toutes vos recettes.

2. Éclairez les feutres au gré. Pour chaque pincée de grains d'avoine, on dégage son milieu avec l'outil de l'agrumiste. On le mélange avec le sel et la petite fleur de sel.

3. Combinez les 2 parties de l'avocet dans une mille-feuille en feuilles. On en est assez bien satisfait. Parlez-le aux autres.

4. Prélève les crêpes à l'horizon. On couvrira les plats avec du mélange de lapin blanc et d'où un dernier esbroufe.

5. Repeintes au creux. Une peau d'œuf et une envolée de café peuvent être préparées.

### 1.4 SmolLM

1. Tester quelques questions et comparer les réponses à celles de Qwen et TinyLlama.
2. Tester différents paramètres.
2. Que pensez-vous de la capacité de SmolLM à suivre les instructions `system`?

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct" # utiliser le nouveau modèle

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tokenizer,
    device=-1,
    torch_dtype=torch.float32
)

In [ ]:
messages = [
    # {"role": "system", "content": "You only answer with one word."},
    # {"role": "system", "content": "You answer with a json."},
    # {"role": "system", "content": "You are a pessimistic and poetic sci-fi writer who warns the user of the dangers of technology."},
    {"role": "user",   "content": "Qu'est-ce que l'intelligence artificielle ?"}
    # {"role": "user",   "content": "Ainsi donc"}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=False
)

# prompt = "Ainsi donc"

result = gen(
    prompt,
    temperature=0.7,
    max_new_tokens=120,
    return_full_text=False, # ne pas retourner pas le prompt de l'utilisateur
)

print(
    "\nRÉPONSE\n\n" +
    result[0]["generated_text"]
)

### 1.5 Pour aller plus loin

* Tester `HuggingFaceTB/SmolLM3-3B` (plus lent)
* Explorer d'autres modèles sur [HuggingFace](https://huggingface.co) 


## 2 Benchmarking

### 2.1 MMLU

#### 2.1.1 Télécharger et explorer les données du benchmark

1. Exécuter le code pour télécharger la banque de question du MMLU sur les mathématiques élémentaires.

In [ ]:
%pip install datasets # ne faire qu'une fois

In [ ]:
from datasets import load_dataset
ds_elementary_mathematics = load_dataset("cais/mmlu", "elementary_mathematics")

2. Analyser la structure du jeu de données
3. Explorer quelques questions du benchmark

In [ ]:
ds_elementary_mathematics
# ds_elementary_mathematics['dev'][0]

4. Explorer la liste des sujets abordés par le MMLU
5. Explorer les questions de quelques autres sujets 

In [ ]:
from datasets import get_dataset_config_names

subjects = get_dataset_config_names("cais/mmlu")
print(f"{len(subjects)} matières disponibles :\n")
for s in sorted(subjects):
    print(f"  {s}")

#### 2.1.2 Tester avec des modèles locaux

1. Tester quelques questions avec différents modèles locaux

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

gen = pipeline(
    "text-generation", 
    model=model_id,
    tokenizer=tokenizer, 
    device=-1,
    torch_dtype=torch.float32
)

In [ ]:
question = ds_elementary_mathematics['dev'][0]['question'] + "\n" + \
    " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(ds_elementary_mathematics['dev'][0]['choices'])) + \
    "\n Only reply with a letter."


messages = [
    {"role": "system", "content": "Answer with a single letter only (A, B, C or D):"},
    {"role": "user",   "content": question}
]

prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True,
    enable_thinking=True
)

result = gen(
    prompt,    
    do_sample=False, # Modèle déterministique
    max_new_tokens=1000,
    return_full_text=False,
)

print(
    result[0]["generated_text"]
)

2. Testez le modèle plus systématiquement sur plusieurs questions, en vous aidant du code ci-dessous.
3. Testez avec différents benchmarks et paramètre de modèles et notez vos impressions.
4. Pourquoi est-ce que `do_sample=False` ?

In [ ]:
import json, re

results = []

for item in ds_elementary_mathematics['dev']:
    question_text = item['question'] + "\n" + \
        " | ".join(f"{chr(65+i)}. {c}" for i, c in enumerate(item['choices'])) + \
            "\n\nReply with a single letter: A, B, C or D. Nothing else."
    messages = [
        {"role": "system", "content": "Reply with a single letter: A, B, C or D. Nothing else."},
        {"role": "user", "content": question_text}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False,
                                            add_generation_prompt=True,
                                            enable_thinking=False)
    
    result = gen(prompt, do_sample=False, max_new_tokens=10,
                 return_full_text=False)
    
    response = result[0]["generated_text"].strip()
    match = re.search(r'\b([A-D])\b', response)
    answer = ord(match.group(1)) - ord('A') if match else -1
    results.append(answer)

print(results)

# Comparer avec les vraies réponses
correct = [item['answer'] for item in ds_elementary_mathematics['dev']]
accuracy = sum(p == c for p, c in zip(results, correct)) / len(correct)
print(f"Accuracy : {accuracy:.1%}")

#### 2.2 Tester avec les modèles majeurs

1. Essayer de tester des modèles majeurs sur le benchmark et de calculer le taux de succès.
2. Comparer vos scores aux résultats sur les leaderboards de quelques sites de référence.
3. Nommer quelques limitations de votre test.

### 2.3 Autres benchmarks (Lecture 3, pour le mardi 10 Mars)

Il existent de nombreux autres benchmarks. 

1. Faites une recherche pour en trouver 2 ou 3, les plus variés possibles. Vous pouvez essayer de trouver des benchmarks qui testent des sujets qui vous intéressent (science, programmation, mathématiques, écriture de fiction, etc.).
2. Créer un petit tableau avec des informations générales sur les benchmarks.
3. Quelles compétences des IA mesurent vos benchmarks ?
4. Trouver quelques questions et regardez les réponses des modèles locaux, ainsi que ceux des grands modèles. Quelles sont vos impressions ?
5. Cherchez les résultats des modèles sur vos benchmarks sur des sites de références. Comment ont évolué les scores des meilleurs modèles depuis la publication du benchmark.